# 🐟 MiroFish Google Colab 1-Click Launcher

Run **MiroFish** completely for free in Google Colab with Cloudflare Tunnel & Google Colab Secrets.

### 🔑 Step 1: Set up your Colab Secrets
Before running the cells, open the **Secrets** panel in the left sidebar (the key icon 🔑) and add:
- `LLM_API_KEY`: Your LLM API key (OpenRouter, Alibaba Bailian, OpenAI, DeepSeek, etc.)
- `ZEP_API_KEY`: Your Zep Cloud API key ([getzep.com](https://app.getzep.com/))
- *(Optional)* `LLM_BASE_URL`: Endpoint (e.g. `https://openrouter.ai/api/v1` or `https://dashscope.aliyuncs.com/compatible-mode/v1`)
- *(Optional)* `LLM_MODEL_NAME`: Model name (e.g. `qwen/qwen-2.5-72b-instruct` or `qwen-plus`)

Make sure to toggle **Notebook access** ON for these secrets!

In [ ]:
# @title 🚀 1. Install System Dependencies & Setup Environment
import os
import sys

# 1. Fetch API Keys from Google Colab Secrets
try:
    from google.colab import userdata
    llm_api_key = userdata.get('LLM_API_KEY')
    zep_api_key = userdata.get('ZEP_API_KEY')
    llm_base_url = userdata.get('LLM_BASE_URL') or 'https://openrouter.ai/api/v1'
    llm_model_name = userdata.get('LLM_MODEL_NAME') or 'qwen/qwen-2.5-72b-instruct'
except Exception as e:
    print('⚠️ Note: Could not automatically load from Colab Secrets: ', e)
    llm_api_key = os.environ.get('LLM_API_KEY', '')
    zep_api_key = os.environ.get('ZEP_API_KEY', '')
    llm_base_url = os.environ.get('LLM_BASE_URL', 'https://openrouter.ai/api/v1')
    llm_model_name = os.environ.get('LLM_MODEL_NAME', 'qwen/qwen-2.5-72b-instruct')

if not llm_api_key or not zep_api_key:
    raise ValueError("❌ Missing required secrets! Please add 'LLM_API_KEY' and 'ZEP_API_KEY' in the Colab Secrets tab (key icon 🔑 on the left) and grant notebook access.")

print('✅ Loaded secrets successfully!')

# 2. Clone repo if not already in workspace
if not os.path.exists('package.json'):
    print('📦 Cloning MiroFish repository...')
    !git clone https://github.com/Jorge-AI700/1MiroFish.git /content/1MiroFish
    %cd /content/1MiroFish

# 3. Write .env file from secrets
env_content = f"""
LLM_API_KEY={llm_api_key}
LLM_BASE_URL={llm_base_url}
LLM_MODEL_NAME={llm_model_name}
ZEP_API_KEY={zep_api_key}
"""
with open('.env', 'w') as f:
    f.write(env_content.strip() + '\n')
print('✅ .env configuration generated!')

# 4. Install uv package manager
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ['PATH'] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"

# 5. Install cloudflared tunnel client
!which cloudflared || (curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared)

# 6. Install Node & Python project dependencies
print('📦 Installing all dependencies (this takes ~1-2 minutes)...')
!npm run setup:all
print('✅ Environment setup complete!')

In [ ]:
# @title 🌐 2. Launch MiroFish & Start Cloudflare Tunnel
import os
import subprocess
import time
import re
from IPython.display import display, HTML

# 1. Clean up any hanging background processes
!pkill -f "run.py" || true
!pkill -f "vite" || true
!pkill -f "cloudflared" || true

# 2. Ensure vite.config.js allows all Cloudflare tunnel hosts
vite_cfg = "frontend/vite.config.js"
if os.path.exists(vite_cfg):
    with open(vite_cfg, "r") as f:
        code = f.read()
    if "allowedHosts" not in code:
        code = code.replace("server: {", "server: {\n    allowedHosts: true,")
        with open(vite_cfg, "w") as f:
            f.write(code)
    print("✅ Verified allowedHosts: true in vite.config.js")

# 3. Ensure frontend API calls use relative path through the tunnel proxy
api_idx = "frontend/src/api/index.js"
if os.path.exists(api_idx):
    with open(api_idx, "r") as f:
        acode = f.read()
    if "'http://localhost:5001'" in acode:
        acode = acode.replace("'http://localhost:5001'", "''")
        with open(api_idx, "w") as f:
            f.write(acode)
    print("✅ Verified frontend API proxy configured!")

# Ensure path includes uv
os.environ['PATH'] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"

print('🚀 Starting MiroFish Backend (port 5001)...')
backend_proc = subprocess.Popen(
    ['npm', 'run', 'backend'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
time.sleep(3)

print('🚀 Starting MiroFish Frontend (port 3000)...')
frontend_proc = subprocess.Popen(
    ['npm', 'run', 'frontend'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
time.sleep(3)

print('🌐 Connecting Cloudflare Tunnel to port 3000...')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:3000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
start_time = time.time()
for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
    if time.time() - start_time > 40:
        break

if public_url:
    display(HTML(f'''
    <div style="padding: 24px; background: linear-gradient(135deg, #059669 0%, #10b981 100%); border-radius: 12px; color: white; font-family: sans-serif; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
        <h2 style="margin: 0 0 12px 0; font-size: 22px; font-weight: bold;">🎉 MiroFish is Live!</h2>
        <p style="margin: 0 0 16px 0; font-size: 15px; opacity: 0.95;">Your simulation sandbox is running in the cloud with zero laptop heat.</p>
        <a href="{public_url}" target="_blank" style="display: inline-block; background-color: white; color: #059669; padding: 12px 24px; border-radius: 8px; font-size: 16px; font-weight: bold; text-decoration: none; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
            👉 Open MiroFish Web UI
        </a>
        <p style="margin: 14px 0 0 0; font-size: 12px; opacity: 0.85;">Public Tunnel URL: {public_url}</p>
    </div>
    '''))
else:
    print('⚠️ Could not extract Cloudflare URL automatically. Showing tunnel logs:')
    for _ in range(10):
        print(tunnel_proc.stdout.readline())

print('\n💡 The services will keep running as long as this Colab cell is active.')
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print('Stopping services...')
    backend_proc.terminate()
    frontend_proc.terminate()
    tunnel_proc.terminate()
